In [18]:
import fitz
import re
import os

## Data exploration

In [9]:
doc = fitz.open("../data/papers/constitutional_ai.pdf")

print(f'Total pages: {len(doc)}')
print('--Page 1 Sample--\n')
print(doc[0].get_text('text')[:500])

Total pages: 34
--Page 1 Sample--

Constitutional AI: Harmlessness from AI Feedback
Yuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion,
Andy Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon,
Carol Chen, Catherine Olsson, Christopher Olah, Danny Hernandez, Dawn Drain,
Deep Ganguli, Dustin Li, Eli Tran-Johnson, Ethan Perez, Jamie Kerr, Jared Mueller,
Jeffrey Ladish, Joshua Landau, Kamal Ndousse, Kamile Lukosuite, Liane Lovitt,
Michael Sellitto, Nelson Elhage, Nicholas Schiefer, Noemi Merc


In [12]:
page = doc[1].get_text('text')
print(f'page 2 word count: {len(page.split())}')
print('--page 2--')
print(page)

page 2 word count: 628
--page 2--
Generate Responses
to “Red Teaming”
Prompts Eliciting Harmful
Samples
Generate Responses
to “Red Teaming”
Prompts Eliciting 
Harmful Samples
 RLAIF
Training
with 
PM + SL-CAI 
Models
Constitutional AI Feedback
for Self-Improvement
Helpful RLHF 
Model
Generate Responses
to “Red Teaming”
Prompts Eliciting Harmful
Samples
Generate Responses
to “Red Teaming”
Prompts Eliciting 
Pairs of Samples
Finetuned
Preference
Model (PM)
Finetuned
SL-CAI
Model
Final
RL-CAI
Model
Response
Critique
Revision
Figure 1
We show the basic steps of our Constitutional AI (CAI) process, which consists of both a super-
vised learning (SL) stage, consisting of the steps at the top, and a Reinforcement Learning (RL) stage, shown
as the sequence of steps at the bottom of the ﬁgure. Both the critiques and the AI feedback are steered by
a small set of principles drawn from a ‘constitution’. The supervised stage signiﬁcantly improves the initial
model, and gives some control over the i

In [16]:
def clean_text(text):
    text = re.sub(r'-\n', '', text)          #fix hyphenated words split across lines
    text = re.sub(r'\n+', ' ', text)         #convert newlines into spaces
    text = re.sub(r'\s{2,}', ' ', text)      #remove extra whitespace

    return text.strip()

    
def chunk_text(text, chunk_size=512, overlap=50):
    target_words = int(chunk_size*0.75)
    overlap_words = int(overlap*0.75)

    words = text.split()
    chunks = []
    start = 0

    while start<len(words):
        end = start + target_words
        chunk = ' '.join(words[start:end])
        if len(chunk.strip())>50:
            chunks.append(chunk)

        start += target_words - overlap_words

    return chunks


cleaned = clean_text(page)
chunks = chunk_text(cleaned)

print(f'Page 2 produced {len(chunks)} chunks\n')
for i, c in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c[:200])
    print()

Page 2 produced 2 chunks

--- Chunk 1 (384 words) ---
Generate Responses to “Red Teaming” Prompts Eliciting Harmful Samples Generate Responses to “Red Teaming” Prompts Eliciting Harmful Samples RLAIF Training with PM + SL-CAI Models Constitutional AI Fee

--- Chunk 2 (278 words) ---
train less harmful systems entirely through the speciﬁcation of a short list of principles or instructions, i.e. a constitution. But we are also employing this terminology to emphasize that when devel



In [19]:
# doing it for the entire dataset
papers_dir = os.path.join('..', 'data', 'papers')
all_chunks = []

for filename in os.listdir(papers_dir):
    if not filename.endswith('.pdf'):
        continue

    path = os.path.join(papers_dir, filename)
    doc = fitz.open(path)
    paper_chunks = []
    
    for page_num in range(len(doc)):
        text = doc[page_num].get_text('text')
        cleaned = clean_text(text)

        if len(cleaned) > 100:
            chunks = chunk_text(cleaned)
            for chunk in chunks:
                paper_chunks.append(
                    {
                        'text': chunk,
                        'source': filename,
                        'page_num': page_num + 1
                    }
                )

    all_chunks.append({'file': filename, 'chunks': paper_chunks})
    print(f'{filename}: {len(paper_chunks)} chunks from {len(doc)} pages')

total = sum(len(p['chunks']) for p in all_chunks)
print(f'\nTotal chunks across all papers: {total}')
                

attention_is_all_you_need.pdf: 26 chunks from 15 pages
concrete_problems_ai_safety.pdf: 61 chunks from 29 pages
constitutional_ai.pdf: 70 chunks from 34 pages
reward_hacking_survey.pdf: 30 chunks from 19 pages
rlhf_christiano.pdf: 34 chunks from 17 pages
scalable_oversight_debate.pdf: 48 chunks from 24 pages

Total chunks across all papers: 269


In [20]:
gitignore_content = """# environment
.env
venv/
__pycache__/
*.pyc

# vector database (can be rebuilt by running ingest)
chroma_db/

# jupyter
.ipynb_checkpoints/
notebooks/.ipynb_checkpoints/

# downloaded papers (can be re-downloaded)
data/papers/*.pdf

# sentence transformer model cache
.cache/
"""

with open("../.gitignore", "w") as f:
    f.write(gitignore_content)

print("created .gitignore")

created .gitignore
